# Walkthrough: from RDP export to all nine figures

This notebook reproduces the analyses in the manuscript end-to-end on a Recovery Data Platform export. It is intended for analysts who want to apply the same recipe to their own RDP data, or for reviewers who want to inspect the analytic decisions at each step.

**What you'll do.** Load an RDP Excel export, build the person-level dataset, decompose the multi-select fields into structured indicators, run Step 1 inferential statistics, run Step 2 machine learning, and produce all nine manuscript figures.

**What you won't do.** Modify the canonical statistical methods. Those choices are documented in the [Developer Guide](../docs/DEVELOPER_GUIDE.md). If you want to swap methods, fork the package.

**Total runtime.** Approximately 2-3 minutes on a laptop. The slowest step is the random forest with permutation importance pooling (about 60 seconds).

---

## Setup

The package should already be installed via `pip install -e .` from the repository root. If not, run that first.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from rdp_analytics import (
    build_person_dataset,
    BernoulliLCA,
    fit_cart_for_pathway,
    fit_random_forest_engagement,
    permutation_importance_pooled,
    umap_projection,
)
from rdp_analytics.features import (
    NEED_DOMAINS, PATHWAY_COMPONENTS, SEVERITY_COMPONENTS,
    build_full_feature_matrix, get_feature_lists,
)
from rdp_analytics.step1_stats import (
    need_pathway_matrix, jonckheere_terpstra, fmt_p,
)
from rdp_analytics.step2_ml import (
    lca_model_selection, reorder_lca_classes_by_complexity,
)
from rdp_analytics.figures import (
    figure_1_distributions, figure_2_heatmap, figure_3_retention,
    figure_4_severity, figure_5_lives_in, figure_6_lca,
    figure_7_cart, figure_8_rf_importance, figure_9_umap,
)

OUT = Path("walkthrough_figures")
OUT.mkdir(exist_ok=True)
SEED = 20260426


## 1. Load the RDP export

The RDP export is an Excel file with one row per assessment record. A participant typically has multiple records (median ~5 in our cohort). The `build_person_dataset` function loads the file, sorts by SAFEID and date, and collapses to one row per participant by taking the most recent non-null value for snapshot fields and aggregating the Reason for Referral field across all records.

Replace the path below with your own RDP export, or download a synthetic dummy export from the repository's `data/sample/` directory (not provided in v0.1.0).

In [ ]:
RDP_PATH = "data/RDP_export.xlsx"  # <-- Update this path

person = build_person_dataset(RDP_PATH)
print(f"Loaded {len(person)} unique participants")
print(f"Columns: {list(person.columns)[:8]}...")  # first 8


## 2. Decompose the multi-select fields into structured indicators

The two RDP fields that carry the most analytic signal are `Reason for Referral` and `Pathways`. Both are semicolon-delimited multi-select fields drawn from controlled vocabularies. We decompose each into a set of binary indicators.

- **Need domains (10 indicators).** Housing, Employment, Education, Treatment connection, Community connection, Recovery support, Probation, Overdose, Co-occurring MH, Multiple treatment episodes.
- **Pathway components (8 indicators).** Abstinence, 12-Step, Support groups, Natural recovery, Peer recovery, MAT, Harm reduction, Alternative/holistic.

We also build a 5-item clinical severity composite and a set of demographic, structural, and service indicators. See [Variable construction glossary](../docs/USER_GUIDE.md#variable-construction-glossary) in the User Guide for the full catalog.

**One thing to call out.** The `intake_complete` indicator is constructed from the presence of any non-null answer to the ethnicity question. This started as a sanity check during debugging and turned out to be the strongest single predictor of engagement in the random forest. We include it explicitly because it's a real signal (data completeness predicts retention) and because it's an easy feature to overlook.

In [ ]:
person = build_full_feature_matrix(person)
feats = get_feature_lists()

# Sanity check the decomposition
print(f"N with need data: {person['need_complexity'].notna().sum()}")
print(f"N with pathway data: {person['pathway_diversity'].notna().sum()}")
print(f"Median need complexity: {person['need_complexity'].median():.0f}")
print(f"Median pathway diversity: {person['pathway_diversity'].median():.0f}")


The analytic dataset for the ML steps restricts to participants with both need and pathway data observed.

In [ ]:
ml = person[person["need_complexity"].notna() & person["pathway_diversity"].notna()].copy()
print(f"Analytic dataset for ML: N = {len(ml)}")


## 3. Step 1: inferential statistics

We're going to use methods that any analyst with introductory biostatistics training can apply. No machine learning yet.

### Figure 1: distributions

Need complexity is right-skewed, with mass at the high end. Pathway diversity is more concentrated, peaking at 3-4 components endorsed simultaneously. This is the empirical case for treating recovery as plural rather than as a binary or ternary choice.

In [ ]:
figure_1_distributions(person, OUT / "fig1_distributions.svg")

# Summary stats
nc = person["need_complexity"].dropna()
pd_s = person["pathway_diversity"].dropna()
print(f"Need complexity:    median={nc.median():.0f}, IQR=[{nc.quantile(.25):.0f}-{nc.quantile(.75):.0f}], mean={nc.mean():.2f}")
print(f"Pathway diversity:  median={pd_s.median():.0f}, IQR=[{pd_s.quantile(.25):.0f}-{pd_s.quantile(.75):.0f}], mean={pd_s.mean():.2f}")

from IPython.display import SVG
SVG(filename=str(OUT / "fig1_distributions.svg"))


### Figure 2: need-pathway adoption matrix with FDR correction

This is the headline Step 1 figure. For each combination of (need domain, pathway component), we ask: among participants who endorse that need at intake, what proportion endorse that pathway? And how does it compare to participants without that need?

We run 80 Fisher's exact tests and correct for multiple comparisons with Benjamini-Hochberg FDR.

In [ ]:
matrix = need_pathway_matrix(person, NEED_DOMAINS, PATHWAY_COMPONENTS)
n_sig = (matrix["qvals"] < 0.05).sum()
print(f"Significant cells at q < 0.05: {n_sig} / 80")
print(f"Significant cells at q < 0.001: {(matrix['qvals'] < 0.001).sum()} / 80")

figure_2_heatmap(matrix, NEED_DOMAINS, PATHWAY_COMPONENTS, OUT / "fig2_heatmap.svg")
SVG(filename=str(OUT / "fig2_heatmap.svg"))


**What to read in this figure.** Two clusters:

- **Structural cluster.** Housing, employment, education, treatment connection, community connection, recovery support → traditional pathways (abstinence, 12-step, support groups, peer recovery).
- **Clinical-severity cluster.** Overdose history, co-occurring MH, multiple treatment episodes, probation → MAT, harm reduction, alternative/holistic. With concomitant DECREASES in abstinence and 12-step endorsement.

This is the heart of the manuscript's empirical contribution.

### Figure 3: retention by need complexity and pathway diversity

We use Jonckheere-Terpstra trend tests rather than Kruskal-Wallis because the exposures are ordinal and we want a directional test.

In [ ]:
figure_3_retention(ml, OUT / "fig3_retention.svg")

# Run the trend tests directly so we can show them inline
nc_levels = sorted(ml["need_complexity"].astype(int).unique())
nc_eng_groups = [ml[ml.need_complexity == k]["engaged"].values for k in nc_levels]
z_eng_nc, p_eng_nc = jonckheere_terpstra(nc_eng_groups)
print(f"Engaged % by need complexity: z = {z_eng_nc:+.2f}, {fmt_p(p_eng_nc)}")

pd_levels = sorted(ml["pathway_diversity"].astype(int).unique())
pd_eng_groups = [ml[ml.pathway_diversity == k]["engaged"].values for k in pd_levels]
z_eng_pd, p_eng_pd = jonckheere_terpstra(pd_eng_groups)
print(f"Engaged % by pathway diversity: z = {z_eng_pd:+.2f}, {fmt_p(p_eng_pd)}")

SVG(filename=str(OUT / "fig3_retention.svg"))


### Figure 4: severity composite × pathway adoption

The 5-item severity composite (history of seizures, lifetime naloxone administration, ER visits, active suicidal ideation, overdose-as-need) drives MAT adoption monotonically from 8% at score 0 to 48% at score 3+.

In [ ]:
figure_4_severity(person, PATHWAY_COMPONENTS, OUT / "fig4_severity.svg")
SVG(filename=str(OUT / "fig4_severity.svg"))


### Figure 5: living situation stratification

This is the recovery capital inversion. Recovery Residence has the LOWEST engagement of any housed category (~61%); Unhoused participants engage at 91%. This contradicts a naive Cox-binary "recovery residence helps retention" framing and is consistent with recovery capital theory: disengagement happens when alternative resources become available.

In [ ]:
figure_5_lives_in(person, OUT / "fig5_lives_in.svg")
SVG(filename=str(OUT / "fig5_lives_in.svg"))


## 4. Step 2: basic machine learning

Now we move to the ML step. Four methods: latent class analysis, classification trees, random forests with permutation importance, and UMAP.

### Figure 6: latent class analysis on need profiles

We fit a Bernoulli mixture model via EM on the 10 need indicators, varying k from 2 to 7 and selecting by BIC.

In [ ]:
X_need = ml[feats["need_features"]].astype(int).values

# Model selection
selection = lca_model_selection(X_need, k_range=range(2, 8), n_init=10, random_state=SEED)
print(selection.to_string(index=False))

best_k = int(selection.loc[selection["BIC"].idxmin(), "k"])
print(f"\nBIC-optimal k: {best_k}")


In [ ]:
# Fit the final model with more restarts for stability
lca = BernoulliLCA(n_classes=best_k, n_init=50, random_state=SEED).fit(X_need)
print(f"Mean modal posterior: {lca.result_.resp.max(axis=1).mean():.3f}")

# Reorder by ascending complexity (sum of item probabilities) so class IDs are interpretable
rho_o, pi_o, resp_o, perm = reorder_lca_classes_by_complexity(
    lca.result_.rho, lca.result_.pi, lca.result_.resp
)
ml["lca_class_ordered"] = resp_o.argmax(axis=1)
ml["lca_post_max"] = resp_o.max(axis=1)

# Render the figure
figure_6_lca(selection, rho_o, pi_o, ml, NEED_DOMAINS, PATHWAY_COMPONENTS, OUT / "fig6_lca.svg")
SVG(filename=str(OUT / "fig6_lca.svg"))


**Reading the LCA figure.** Panel B is the heart of it. Each row is a class. Each column is a need domain. The number is the probability that someone in that class endorsed that domain. Class C5 (high-complexity clinical) endorses essentially every domain; class C1 (single-need recovery support) endorses Recovery support and almost nothing else.

### Figure 7: classification tree for MAT pathway endorsement

A depth-4 decision tree predicts MAT endorsement with cross-validated AUC ~0.71. The top split is lifetime naloxone history.

In [ ]:
cart, cart_info = fit_cart_for_pathway(
    ml,
    features=feats["features_for_pathway"],
    target_col="path_Medication_assisted_Recovery",
    max_depth=4,
    random_state=SEED,
)
print(f"CART CV AUC: {cart_info['cv_auc_mean']:.3f} ± {cart_info['cv_auc_std']:.3f}")

figure_7_cart(cart, cart_info["features"], OUT / "fig7_cart.svg")
SVG(filename=str(OUT / "fig7_cart.svg"))


### Figure 8: random forest with permutation importance

Random forest, 200 trees, class-balanced, predicting engagement at last observation.

In [ ]:
rf, rf_info = fit_random_forest_engagement(
    ml, features=feats["features_for_engagement"], random_state=SEED,
)
print(f"RF CV AUC: {rf_info['cv_aucs'].mean():.3f} ± {rf_info['cv_aucs'].std():.3f}")
print(f"RF OOB:    {rf_info['oob']:.3f}")

# Pooled permutation importance across 4 seeds (effective n_repeats=20)
data = ml[feats["features_for_engagement"] + ["engaged"]].dropna()
X = data[feats["features_for_engagement"]].values
y = data["engaged"].values

perm_means, perm_stds = permutation_importance_pooled(rf, X, y)

# Top 10 features
order = perm_means.argsort()[::-1]
print("\nTop 10 features by permutation importance:")
for r, i in enumerate(order[:10]):
    print(f"  {r+1:2d}. {feats['features_for_engagement'][i]:35s}  {perm_means[i]:+.4f} ± {perm_stds[i]:.4f}")


In [ ]:
figure_8_rf_importance(rf_info, perm_means, perm_stds, feats["features_for_engagement"], OUT / "fig8_rf_importance.svg")
SVG(filename=str(OUT / "fig8_rf_importance.svg"))


**Two findings to note.**

- **Operational.** Intake completion is the strongest single predictor. Participants who complete the intake form are 79% engaged versus 40% in those who don't. This is a process variable, not a clinical one, but it has direct operational implications.
- **Uncomfortable.** Has Peer match ranks 26th of 42 features. Variation in peer-match status does not explain variation in engagement in this cohort. This is not "peer matching doesn't matter"; it is "we cannot detect a peer-matching effect after controlling for everything else."

### Figure 9: UMAP projection

Project participants into 2D using UMAP on the standardized 34-feature matrix (excluding pathway components to avoid trivial circularity with LCA classes).

In [ ]:
umap_features = [f for f in feats["features_for_engagement"] if not f.startswith("path_")]
emb, umap_data = umap_projection(ml, umap_features, random_state=SEED)
print(f"UMAP embedding shape: {emb.shape}")

figure_9_umap(emb, umap_data, ml, OUT / "fig9_umap.svg")
SVG(filename=str(OUT / "fig9_umap.svg"))


## 5. Wrap up

You should now have all nine manuscript figures in `walkthrough_figures/` plus the intermediate datasets and model artifacts.

### A note on what we didn't do

A few things this notebook deliberately doesn't include:

- **Sensitivity analyses.** The complete-case severity composite, the pathway-non-switching diagnostic, and the alternative-k LCA fits are all worth running. See `pipeline.py --sensitivity` (placeholder) or implement them as additional cells here.
- **Causal claims.** The data is observational. We describe associations and predictions, not causes.
- **Recommendations.** The decision tree (Figure 7) is a decision aid, not a recommendation engine. Participants decide their pathway. The model surfaces patterns that let peer specialists not be surprised.

### What to do next

1. If you're applying this to a different RDP export and the numbers are very different from ours, that's interesting and worth writing up. Open an issue on the repository.
2. If a figure looks subtly wrong, check the variable construction glossary in the [User Guide](../docs/USER_GUIDE.md). Most surprising results trace back to a substring-matching bug or a missingness assumption.
3. If you found a bug or want to extend the package, see [CONTRIBUTING.md](../CONTRIBUTING.md).

### Citation

If you use this analysis in published work, cite the manuscript:

> Walton CM, Oldham BB, Markie F, Fiala B, Bell MS. Real-world decision trees for peer recovery support specialists: findings and methods from 1,411 community peer recovery participants in Minnesota. *Manuscript in preparation, 2026.*